# Create Other RES Dataset

Reads the **Other RES** sheet from every PEMMDB node file and extracts:
1. **Timeseries** (`other_res_timeseries.csv`) — rows = hourly timestamps, columns = `{node}_{source_type}` with MW output values
2. **Capacities** (`other_res_capacities.csv`) — rows = (node, source_type, installed_capacity_MW)

Sheet structure:
- Row 8, cols E–I: source type names (Small Biomass, Geothermal, Marine, Waste, Not Defined)
- Row 9, cols E–I: installed capacity (MW)
- Rows 11–8770: hourly output (MW) — col C = total, cols E–I = per source


In [ ]:
import pandas as pd
import openpyxl
from pathlib import Path
from tqdm.notebook import tqdm

PEMMDB_DIR = Path("../data/tyndp2024/PEMMDB2/2030")
SHEET      = "Other RES"

# Source type columns: E=4, F=5, G=6, H=7, I=8  (0-indexed)
SOURCE_COLS = [4, 5, 6, 7, 8]

# Output containers
timeseries_dict = {}   # key: "{node}_{source}" → pd.Series (8760 values)
capacities_rows = []   # list of dicts: node, source_type, installed_capacity_mw

files = sorted(PEMMDB_DIR.glob("PEMMDB_*_NationalTrends_2030.xlsx"))
print(f"Found {len(files)} PEMMDB files")

for fpath in tqdm(files, desc="Reading Other RES"):
    # Extract node name from filename: PEMMDB_DE00_NationalTrends_2030.xlsx → DE00
    node = fpath.stem.split("_")[1]

    wb = openpyxl.load_workbook(fpath, read_only=True, data_only=True)

    if SHEET not in wb.sheetnames:
        wb.close()
        continue

    ws   = wb[SHEET]
    rows = list(ws.iter_rows(values_only=True))
    wb.close()

    # Row 8 (index 7): source type names in cols E–I 
    header_row = rows[7]   # row 8, 0-indexed = 7
    source_types = [str(header_row[c]).strip() if header_row[c] is not None else f"col_{c}"
                    for c in SOURCE_COLS]

    # Row 9 (index 8): installed capacities in cols E–I 
    cap_row = rows[8]      # row 9
    for c, stype in zip(SOURCE_COLS, source_types):
        cap_val = cap_row[c]
        cap_mw  = float(cap_val) if cap_val is not None else 0.0
        capacities_rows.append({
            "node":                  node,
            "source_type":           stype,
            "installed_capacity_mw": cap_mw,
        })

    # Rows 11–8770 (index 10–8769): hourly output in cols E–I 
    data_rows = rows[10:10 + 8760]   # exactly 8760 hours
    if len(data_rows) < 8760:
        print(f"  ⚠ {node}: only {len(data_rows)} data rows, skipping timeseries")
        continue

    for c, stype in zip(SOURCE_COLS, source_types):
        col_key = f"{node}_{stype}"
        values  = [float(r[c]) if r[c] is not None else 0.0 for r in data_rows]
        timeseries_dict[col_key] = values

print(f"\nDone. Collected {len(timeseries_dict)} timeseries columns across all nodes.")
print(f"Capacities rows: {len(capacities_rows)}")


Found 77 PEMMDB files


Reading Other RES:   0%|          | 0/77 [00:00<?, ?it/s]


Done. Collected 385 timeseries columns across all nodes.
Capacities rows: 385


In [ ]:
# Build datetime index (8760 h, 2030, first Jan = Monday per PEMMDB convention) 
timestamps = pd.date_range("2030-01-01", periods=8760, freq="h")

# 1. Timeseries DataFrame 
df_ts = pd.DataFrame(timeseries_dict, index=timestamps)
df_ts.index.name = "time"

# Drop columns that are all-zero (source types with no capacity anywhere)
nonzero_cols = df_ts.columns[(df_ts != 0).any()]
df_ts = df_ts[nonzero_cols]

print(f"Timeseries shape: {df_ts.shape}  (rows=hours, cols=node_source)")
print(f"Non-zero columns: {len(nonzero_cols)}")
print()
print(df_ts.head(3))

# 2. Capacities DataFrame
df_cap = pd.DataFrame(capacities_rows)
df_cap = df_cap[df_cap.installed_capacity_mw > 0].reset_index(drop=True)

print(f"\nCapacities shape: {df_cap.shape}")
print(df_cap.head(10).to_string())

# Save
out_ts  = Path("../data/open-tyndp/other_res_timeseries.csv")
out_cap = Path("../data/open-tyndp/other_res_capacities.csv")

df_ts.to_csv(out_ts)
df_cap.to_csv(out_cap, index=False)

print(f"\n✓ Saved timeseries  → {out_ts}")
print(f"✓ Saved capacities  → {out_cap}")


Timeseries shape: (8760, 93)  (rows=hours, cols=node_source)
Non-zero columns: 93

                     AT00_Small Biomass  AT00_Waste  BE00_Small Biomass  \
time                                                                      
2030-01-01 00:00:00          427.417629   74.531957          288.215618   
2030-01-01 01:00:00          427.417629   74.531957          285.359238   
2030-01-01 02:00:00          427.417629   74.531957          284.763294   

                     BE00_Waste  BG00_Small Biomass  \
time                                                  
2030-01-01 00:00:00   24.415353          121.584416   
2030-01-01 01:00:00   24.173383          117.662338   
2030-01-01 02:00:00   24.122900          121.584416   

                     CH00_Not Defined / Splitting not known  \
time                                                          
2030-01-01 00:00:00                              185.176586   
2030-01-01 01:00:00                              185.176586   
2030-01-01 02

In [ ]:
# Aggregate per node: sum all source types → {node}_other-res 
# p_max_pu = total_output_MW / total_installed_capacity_MW  (per node, per hour)

# All unique nodes in the timeseries
all_nodes = sorted({col.split("_")[0] for col in df_ts.columns})

p_max_pu_dict = {}   # key: node → 8760-length Series of p_max_pu values
agg_rows      = []   # for the aggregated capacity CSV

for node in all_nodes:
    # Sum all source-type columns for this node
    node_cols    = [c for c in df_ts.columns if c.startswith(f"{node}_")]
    total_output = df_ts[node_cols].sum(axis=1)   # MW, 8760 h

    # Total installed capacity for this node (sum across source types)
    total_cap_mw = df_cap[df_cap.node == node].installed_capacity_mw.sum()

    if total_cap_mw > 0:
        p_max_pu = (total_output / total_cap_mw).clip(0.0, 1.0)
    else:
        p_max_pu = pd.Series(0.0, index=df_ts.index)

    p_max_pu_dict[node] = p_max_pu

    agg_rows.append({
        "node":                  node,
        "carrier":               "other-res",
        "installed_capacity_mw": total_cap_mw,
        "annual_twh":            total_output.sum() / 1e6,
        "mean_p_max_pu":         p_max_pu.mean(),
    })

# Build aggregated p_max_pu DataFrame 
df_pmax = pd.DataFrame(p_max_pu_dict, index=df_ts.index)
df_pmax.index.name = "time"
df_pmax.columns = [f"{node}" for node in df_pmax.columns]  # just node name as col

df_agg_cap = pd.DataFrame(agg_rows)

print(f"Aggregated p_max_pu shape: {df_pmax.shape}  (rows=hours, cols=nodes)")
print(f"\nSample — DE00:")
print(f"  p_max_pu  min={df_pmax['DE00'].min():.3f}  mean={df_pmax['DE00'].mean():.3f}  max={df_pmax['DE00'].max():.3f}")
print()
print(df_agg_cap[df_agg_cap.node.isin(["DE00", "FR00", "GB00", "PL00", "ES00"])].to_string(index=False))

# Save 
out_pmax    = Path("../data/open-tyndp/other_res_p_max_pu.csv")
out_agg_cap = Path("../data/open-tyndp/other_res_agg_capacities.csv")

df_pmax.to_csv(out_pmax)
df_agg_cap.to_csv(out_agg_cap, index=False)

print(f"\n✓ Saved p_max_pu timeseries → {out_pmax}")
print(f"✓ Saved agg capacities      → {out_agg_cap}")


Aggregated p_max_pu shape: (8760, 45)  (rows=hours, cols=nodes)

Sample — DE00:
  p_max_pu  min=0.615  mean=0.615  max=0.615

node   carrier  installed_capacity_mw  annual_twh  mean_p_max_pu
DE00 other-res           13067.586000   70.367197       0.614711
ES00 other-res            1730.000200    7.677783       0.506624
FR00 other-res            2390.000000   12.059454       0.576004
PL00 other-res            2155.053722   10.473315       0.554782

✓ Saved p_max_pu timeseries → ../data/open-tyndp/other_res_p_max_pu.csv
✓ Saved agg capacities      → ../data/open-tyndp/other_res_agg_capacities.csv

✓ Saved p_max_pu timeseries → ../data/open-tyndp/other_res_p_max_pu.csv
✓ Saved agg capacities      → ../data/open-tyndp/other_res_agg_capacities.csv


In [ ]:
# Quick validation: check DE00 
print("=== DE00 Other RES validation ===")

de_cols = [c for c in df_ts.columns if c.startswith("DE00_")]
print(f"Columns: {de_cols}")
print()

for col in de_cols:
    total_twh = df_ts[col].sum() / 1e6
    cap_row   = df_cap[(df_cap.node == "DE00") & (df_cap.source_type == col.replace("DE00_", ""))]
    cap_mw    = cap_row.installed_capacity_mw.values[0] if len(cap_row) > 0 else 0
    cf        = df_ts[col].mean() / cap_mw if cap_mw > 0 else 0
    print(f"  {col:<45}  cap={cap_mw:>8,.1f} MW   annual={total_twh:.2f} TWh   CF={cf:.3f}")

print()
total_de = df_ts[[c for c in de_cols]].sum(axis=1)
print(f"  DE00 total other-res annual: {total_de.sum()/1e6:.1f} TWh")
print(f"  DE00 total installed cap   : {df_cap[df_cap.node=='DE00'].installed_capacity_mw.sum():,.0f} MW")

# Compare with TYNDP target
print()
print("  TYNDP 2030 DE target: ~70 TWh")
print(f"  PEMMDB raw data sum : {total_de.sum()/1e6:.1f} TWh  ← this is actual dispatch profile")
print(f"  CF of total         : {total_de.mean() / df_cap[df_cap.node=='DE00'].installed_capacity_mw.sum():.3f}")


=== DE00 Other RES validation ===
Columns: ['DE00_Small Biomass', 'DE00_Waste', 'DE00_Not Defined / Splitting not known']

  DE00_Small Biomass                             cap=10,000.0 MW   annual=59.65 TWh   CF=0.681
  DE00_Waste                                     cap= 2,297.1 MW   annual=8.05 TWh   CF=0.400
  DE00_Not Defined / Splitting not known         cap=   770.5 MW   annual=2.67 TWh   CF=0.395

  DE00 total other-res annual: 70.4 TWh
  DE00 total installed cap   : 13,068 MW

  TYNDP 2030 DE target: ~70 TWh
  PEMMDB raw data sum : 70.4 TWh  ← this is actual dispatch profile
  CF of total         : 0.615
